# Stat 220, Unit 4 Homework: Prediction and Choosing Predictors

`delivery_routes.csv` is 500 completed delivery routes. `minutes` is how long the
route took. The predictors that make sense are `stops`, `packages`, `miles`, `downtown`, and
`rain`. The company also records `van_age_years`, `dispatcher_rating`, and `month`.

Run the next two cells first. The first one installs what you need, and the second loads the
data that everything else here uses.

In [ ]:
%pip install -q numpy pandas statsmodels scikit-learn matplotlib

In [5]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_score, train_test_split

routes = pd.read_csv("https://drbob-richardson.github.io/stat220/F2026/data/delivery_routes.csv")
folds = KFold(5, shuffle=True, random_state=0)
routes.head()

,minutes,stops,packages,miles,downtown,rain,van_age_years,dispatcher_rating,month
0,254.3,49,105,55.6,1,0,8,3,7
1,153.5,24,71,32.0,0,0,5,5,2
2,450.9,73,226,55.4,1,1,7,4,4
3,387.1,71,175,73.2,1,0,1,1,12
4,111.2,12,18,32.1,0,1,7,1,11


**Problem 1.** *Planning tomorrow's schedule.* Tomorrow's route has 45 stops, 110 packages,
38 miles, goes downtown, and no rain is forecast.

Part a. Report the predicted minutes, the confidence interval, and the prediction interval. Say in one sentence what each of the two intervals is about.

In [7]:
fit = smf.ols("minutes ~ stops + packages + miles + downtown + rain", data=routes).fit()

new = pd.DataFrame({"stops": [45], "packages": [110], "miles": [38],
                    "downtown": [1], "rain": [0]})
print(fit.get_prediction(new).summary_frame(alpha=0.05).round(1))

    mean  mean_se  mean_ci_lower  mean_ci_upper  obs_ci_lower  obs_ci_upper
0  273.9      2.1          269.7          278.1         229.9         317.8


_Your answer:_ Considering the variables affecting tomorrow's route, we can assume that it will take between 229.9 and 317.8 minutes (prediction interval) with a good estimate of 273.9 minutes (95% CI: 269.7, 278.1).

The confidence interval provides us with a possible range of values based on the mean of previous trips while the prediction interval provides a wider range that accounts for outliers, helping us know what to expect from a single new observation.



Part b. The dispatcher has to tell one driver when to expect to finish tomorrow. Which of the two intervals should they use, and why?

_Your answer:_ The dispatcher should tell the driver to use the prediction interval, because it accounts for any fluctuation that might increase or decrease the time needed. Since the prediction interval is wider, this would also help to manage expectations for that driver.



Part c. The operations manager asks a different question: across all the downtown routes like this one, what is our average time? Which interval answers that one?

In [8]:
from scipy import stats

downtown_routes = routes[routes.downtown == 1]

n = downtown_routes.minutes.count()
mean = downtown_routes.minutes.mean()
se = downtown_routes.minutes.sem()  # standard error of the mean

ci_low, ci_high = stats.t.interval(0.95, df=n - 1, loc=mean, scale=se)

print(f"n = {n}, mean = {mean:.1f}, 95% CI = ({ci_low:.1f}, {ci_high:.1f})")

n = 164, mean = 284.0, 95% CI = (267.0, 301.0)


_Your answer:_ Accounting purely for downtown routes regardless of other variables, we would expect an average time of 284 minutes and would use the mean confidence interval (95% CI: 267.0, 301.0). If we were to hold all other variables constant and assume that all downtown routes are "like this one" we would expect an average time of 273.9 and would use the mean interval (95% CI: 269.7, 278.1).



**Problem 2.** *A route that looks ordinary.* A planner asks for a prediction for a route with
25 stops and 120 packages.

Part a. Each value is common on its own. Report how many completed routes are near each value, and how many are near both.

In [10]:
near_stops = routes.stops.between(20, 30)
near_packages = routes.packages.between(110, 130)

print("routes near this stop count   :", near_stops.sum())
print("routes near this package count:", near_packages.sum())
print("routes near both              :", (near_stops & near_packages).sum())
print("correlation between stops and packages:", round(routes.stops.corr(routes.packages), 2))

routes near this stop count   : 86
routes near this package count: 59
routes near both              : 0
correlation between stops and packages: 0.92


_Your answer:_ There are 86 completed routes near 25 stops (between 20-30) and there are 59 routes with close to 120 packages (between 110-130), but there are 0 (zero) routes that have both of these attributes.



Part b. In three or four sentences, explain what your part a numbers mean for a prediction for this route, and what you would tell the planner who asked for it.

_Your answer:_ While there are many routes (86) with the requested number of stops and there are many routes (59) with the requested package count, there is no occurance of both in our dataset. This is because there is a strong correlation between stop count and package count, so we would expect more stops or fewer packages. Considering the lack of data for a route with 25 stops and 120 packages we would not be able to create an accurate prediction for this route using the data available to us.



**Problem 3.** *How wrong will it be?* A new depot has 50 completed routes so far, and the
manager wants one number for how far off a time estimate typically is.

Part a. Report both numbers, and say which one you would give the manager.

In [11]:
depot = routes.sample(50, random_state=3)
X = depot[["stops", "packages", "miles", "downtown", "rain"]]
y = depot["minutes"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)
m = LinearRegression().fit(X_train, y_train)

print("on the rows it was fitted on:", round(np.sqrt(((y_train - m.predict(X_train))**2).mean()), 1))
print("on the rows held back       :", round(np.sqrt(((y_test - m.predict(X_test))**2).mean()), 1))

on the rows it was fitted on: 22.9
on the rows held back       : 27.7


_Your answer:_ With 50 completed routes, we see that the RMSE on rows fitted to the model was 22.9 minutes and RMSE on rows held back was 27.7 minutes. I would give the manager the error for the held back rows (27.7), using the same logic as the previous confidence interval selection: the variance of a new addition could be wider than a narrower prediction may include.



Part b. Explain in two or three sentences why those two numbers differ, and which way the difference would go if the depot had 500 routes instead of 50.

_Your answer:_ The two numbers are different because errors measured on the data it is fitted on would be overly confident. Using the model on the un-fitted data gives us a real field test of what errors would look like for new data. If we were to have 500 routes instead of 50, the numbers would likely be much closer if not the same because we would have enough data to account for most potential outliers.



**Problem 4.** *Which predictors belong?* Still the 50-route depot.

Part a. Report the cross-validated error, the $R^2$, and the AIC for each set. Say which set you would ship.

In [12]:
sets = {
    "stops only": ["stops"],
    "the five that make sense": ["stops", "packages", "miles", "downtown", "rain"],
    "those five plus three junk": ["stops", "packages", "miles", "downtown", "rain",
                                   "van_age_years", "dispatcher_rating", "month"],
}
for name, cols in sets.items():
    mse = -cross_val_score(LinearRegression(), depot[cols], depot.minutes, cv=folds,
                           scoring="neg_mean_squared_error").mean()
    f = smf.ols("minutes ~ " + " + ".join(cols), data=depot).fit()
    print(f"{name:<28} CV error {np.sqrt(mse):5.1f}   R2 {f.rsquared:.4f}   AIC {f.aic:.1f}")

stops only                   CV error  38.1   R2 0.9165   AIC 503.7
the five that make sense     CV error  27.3   R2 0.9625   AIC 471.7
those five plus three junk   CV error  31.1   R2 0.9646   AIC 474.8


_Your answer:_ Using the same 50-route data set, accounting for stops only we see a CV error of 38.1, an R^2 of 0.9165, and AIC of 503.7. Accounting for the five most relevant variables gives us a CV error of 27.3, R^2 of 0.9625, and AIC of 417.7. Accounting for the five most relevant variables and three additional "junk" variables gives us a CV error of 31.1, R^2 of 0.9646, and AIC of 474.8.

Given this information we would ship the model using the five most relevant variables. We select this because it has the lowest error (27.3), a strong R^2 (0.96), and the lowest AIC (471.7).



Part b. One set has the highest $R^2$ and is still not the set you would ship. Explain what $R^2$ is doing here in two or three sentences.

_Your answer:_ The set with the highest R^2 is the set using the five most relevant variables and three additional junk variables. The reason for this increase is that R^2 will always increase with additional predictors, so we next look to AIC to determine which set is the most appropriate, and in this case the difference between R^2 for both sets is negligible.



Part c. A colleague wants to add a squared term in `stops` on top of the five. Say what you would check before agreeing, and what you would expect with only 50 routes.

_Your answer:_ Before agreeing to add the squared term in stops to the model, I would want to check the cross-validated error and AIC for meaningful improvement--R^2 I would expect to increase with the added predictor. There would need to be improvements in those other tests to justify the more complicated model, especially with only 50 routes in our dataset which might be more susceptible to unintentional fitted noise with a squared variable.



**Problem 5.** *What can and cannot be said.* No computer.

Part a. A colleague ran stepwise selection over 65 columns, kept the 11 with p-values under 0.05, and reports an $R^2$ of 0.96 with every predictor significant. Write the two or three sentences you would say in response.

_Your answer:_ It sounds like my colleague took care to not compromise their findings. The p-values and R^2 are both indications of this, but 11 predictors still might be too many, I would suggest running an additional backwards selection on what is left to see if any of the 11 significant variables are the best predictors or if some of them are simply supporting each other.



Part b. A lasso fit sends `rain` to exactly zero. Does that show rain has no effect on how long a route takes? Explain.

_Your answer:_ The lasso fit sending 'rain' to exactly zero does not show that rain has no effect on how long a route takes, but is instead expected behavior of a lasso. Lasso fit sends all variables to exactly zero at some point, so the real impact is to have a reasonable cutoff that only excludes the smallest predictors. Comparing 'rain' to other predictors using a lasso fit and a reasonable cutoff still might exclude rain, but even then it would not be an indication that rain has no effect, but that it's effect is less relevant than other predictors that we would like to explore more.



Part c. The model predicts well and `packages` has a large coefficient. The manager asks whether splitting the same deliveries into more packages would make routes take longer. Answer in two or three sentences.

_Your answer:_ Given that 'packages' has a large coefficient, it is reasonable to assume that adding additional packages to our deliveries would make the routes take longer. The model predicts well, so we can assume that this coefficient holds 'stops' and other variables constant and is therefore a reliable estimation of the time added by isolating the 'packages' variable and adding to it.

